In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("../../data/creditcard.csv")

In [3]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO TREINO/TESTE
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
from imblearn.combine import SMOTEENN
from collections import Counter

def apply_smoteenn(X_train, y_train):
    sme = SMOTEENN(random_state=42)
    X_train, y_train = sme.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

In [5]:
from imblearn.over_sampling import ADASYN

def apply_adasyn(X_train, y_train):
    adasyn = ADASYN(random_state=42)
    X_train, y_train = adasyn.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

In [6]:
from imblearn.over_sampling import BorderlineSMOTE

def apply_borderline_smote(X_train, y_train):
    borderline_smote = BorderlineSMOTE(random_state=42)
    X_train, y_train = borderline_smote.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

In [7]:
from imblearn.over_sampling import KMeansSMOTE

def apply_kmeans_smote(X_train, y_train):
    kmeans_smote = KMeansSMOTE(random_state=42)
    X_train, y_train = kmeans_smote.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

In [8]:
X_train_original, y_train_original = X_train.copy(), y_train.copy()

In [9]:
# =========================
# INSTALAR PYCARET
# =========================
#%pip install pycaret

# =========================
# IMPORTS
# =========================
import pandas as pd

from sklearn.model_selection import train_test_split
from pycaret.classification import setup, compare_models, predict_model, pull
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


for resampling_method in ['kmeans_smote']:
    print(f"\n===== MÉTODO DE REAMOSTRAGEM: {resampling_method.upper()} =====")
    if resampling_method == 'smoteenn':
        X_train, y_train = apply_smoteenn(X_train_original, y_train_original)
    elif resampling_method == 'adasyn':
        X_train, y_train = apply_adasyn(X_train_original, y_train_original)
    elif resampling_method == 'borderline_smote':
        X_train, y_train = apply_borderline_smote(X_train_original, y_train_original)
    elif resampling_method == 'kmeans_smote':
        try:
            X_train, y_train = apply_kmeans_smote(X_train_original, y_train_original)
        except RuntimeError as e:
            if "No clusters found with sufficient samples" not in str(e):
                raise  # Let other RuntimeErrors surface
            print(f"Skipping {resampling_method}: {e}")
            continue
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PYCARET
    # =========================

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    exp = setup(
        data=train_df,
        test_data=test_df,
        target="Class",
        index=False,  # PyCaret assigns a fresh index
        session_id=42,
        verbose=False
    )


    # Retorna os 10 melhores modelos treinados
    modelos_treinados = compare_models(n_select=10)

    # =========================
    # LEADERBOARD INTERNO DO PYCARET
    # =========================
    print("\n===== LEADERBOARD (Validação PyCaret) =====")
    print(pull())

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS (No Test Set)
    # =========================
    resultados = []

    for model in modelos_treinados:
        model_name = model.__class__.__name__

        # Previsão no conjunto de teste
        preds_df = predict_model(model, data=test_df, verbose=False)

        y_true = test_df['Class'].values
        # No PyCaret 3.x, a coluna com o rótulo previsto é 'prediction_label'
        y_pred = preds_df['prediction_label'].astype(int).values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted
        })

    # =========================
    # RESULTADOS
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
    print(df_resultados.sort_values(by="F1-score Weighted", ascending=False))


===== MÉTODO DE REAMOSTRAGEM: KMEANS_SMOTE =====
Skipping kmeans_smote: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


In [10]:
# =========================
# INSTALAR H2O
# =========================
%pip install h2o

# =========================
# IMPORTS
# =========================
import pandas as pd
import h2o
from h2o.automl import H2OAutoML

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# INICIAR H2O
# =========================
h2o.init()

for resampling_method in ['smoteenn', 'adasyn', 'borderline_smote', 'kmeans_smote']:
    print(f"\n===== MÉTODO DE REAMOSTRAGEM: {resampling_method.upper()} =====")
    if resampling_method == 'smoteenn':
        X_train, y_train = apply_smoteenn(X_train_original, y_train_original)
    elif resampling_method == 'adasyn':
        X_train, y_train = apply_adasyn(X_train_original, y_train_original)
    elif resampling_method == 'borderline_smote':
        X_train, y_train = apply_borderline_smote(X_train_original, y_train_original)
    elif resampling_method == 'kmeans_smote':
        X_train, y_train = apply_kmeans_smote(X_train_original, y_train_original)

    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PARA H2O
    # =========================
    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    train_h2o['Class'] = train_h2o['Class'].asfactor()
    test_h2o['Class'] = test_h2o['Class'].asfactor()

    x = train_h2o.columns[:-1]
    y = 'Class'

    # =========================
    # AUTO ML
    # =========================
    aml = H2OAutoML(
        max_models=20,
        seed=42,
        balance_classes=True
    )

    aml.train(x=x, y=y, training_frame=train_h2o)

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    print(aml.leaderboard)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    leaderboard = aml.leaderboard.as_data_frame()

    resultados = []

    for model_id in leaderboard['model_id']:

        model = h2o.get_model(model_id)

        # previsão
        preds = model.predict(test_h2o).as_data_frame()

        y_true = test_df['Class'].values
        y_pred = preds['predict'].astype(int).values

        # métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

        resultados.append({
            "Modelo": model_id,
            "Tipo": model.algo,
            "Accuracy": round(acc, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1-score": round(f1, 4),
            "F1-score Weighted": round(f1_weighted, 4)
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
    print(df_resultados.sort_values(by="F1-score Weighted", ascending=False))

Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,9 days 23 hours 16 mins
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 4 days
H2O_cluster_name:,H2O_from_python_lannaces_gfr9u3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2.783 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+

===== MÉTODO DE REAMOSTRAGEM: SMOTEENN =====
Resampled dataset shape Counter({1: 218370, 0: 209282})
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
21:29:03.10: AutoML: XGBoost is not available; skipping it.

██████

RuntimeError: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.

In [ ]:
# =========================
# INSTALAR LAZYPREDICT
# =========================
#%pip install lazypredict

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lazypredict.Supervised import LazyClassifier

for resampling_method in ['smoteenn', 'adasyn', 'borderline_smote', 'kmeans_smote']:
    print(f"\n===== MÉTODO DE REAMOSTRAGEM: {resampling_method.upper()} =====")
    if resampling_method == 'smoteenn':
        X_train, y_train = apply_smoteenn(X_train_original, y_train_original)
    elif resampling_method == 'adasyn':
        X_train, y_train = apply_adasyn(X_train_original, y_train_original)
    elif resampling_method == 'borderline_smote':
        X_train, y_train = apply_borderline_smote(X_train_original, y_train_original)
    elif resampling_method == 'kmeans_smote':
        X_train, y_train = apply_kmeans_smote(X_train_original, y_train_original)

    # =========================
    # LAZYPREDICT
    # =========================
    clf = LazyClassifier(verbose=0, ignore_warnings=True)

    models, predictions = clf.fit(X_train, X_test, y_train, y_test)

    # =========================
    # RESULTADOS
    # =========================
    print("\n===== RESULTADOS =====")
    print(models)

    print("\n===== MÉTRICAS =====")
    all_metrics = []

    for model_name, model in clf.models.items():
        y_pred = model.predict(X_test)

        all_metrics.append({
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "F1 Score Weighted": f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'),
            "F1 Score": f1_score(y_test, y_pred, pos_label=1, zero_division=0)

        })

    metrics_df = (
        pd.DataFrame(all_metrics)
        .sort_values("F1 Score Weighted", ascending=False)
        .reset_index(drop=True)
    )

    print(metrics_df)

In [11]:
# =========================
# INSTALAR FLAML
# =========================
%pip install "flaml[automl]"

# =========================
# IMPORTS
# =========================
from flaml import AutoML

for resampling_method in ['smoteenn', 'adasyn', 'borderline_smote', 'kmeans_smote']:
    print(f"\n===== MÉTODO DE REAMOSTRAGEM: {resampling_method.upper()} =====")
    if resampling_method == 'smoteenn':
        X_train, y_train = apply_smoteenn(X_train_original, y_train_original)
    elif resampling_method == 'adasyn':
        X_train, y_train = apply_adasyn(X_train_original, y_train_original)
    elif resampling_method == 'borderline_smote':
        X_train, y_train = apply_borderline_smote(X_train_original, y_train_original)
    elif resampling_method == 'kmeans_smote':
        X_train, y_train = apply_kmeans_smote(X_train_original, y_train_original)


    # =========================
    # TREINAMENTO COM FLAML
    # =========================
    automl = AutoML()

    settings = {
        "time_budget": 60,
        "metric": 'accuracy',
        "task": 'classification',
        "log_file_name": "flaml.log",
    }

    automl.fit(X_train=X_train, y_train=y_train, **settings)

    # =========================
    # RESULTADOS
    # =========================
    print("\n===== RESULTADOS =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # AVALIAÇÃO
    # =========================
    y_pred = automl.predict(X_test)

    print("\n===== MÉTRICAS =====")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
    print("F1 Score Weighted:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))
    print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4))

Note: you may need to restart the kernel to use updated packages.

===== MÉTODO DE REAMOSTRAGEM: SMOTEENN =====
Resampled dataset shape Counter({1: 218370, 0: 209282})
[flaml.automl.logger: 10-16 08:11:14] {2470} INFO - task = classification
[flaml.automl.logger: 10-16 08:11:14] {2481} INFO - Evaluation method: holdout
[flaml.automl.logger: 10-16 08:11:14] {2584} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 10-16 08:11:14] {2701} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 10-16 08:11:14] {3006} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 10-16 08:11:15] {3141} INFO - Estimated sufficient time budget=38125s. Estimated necessary time budget=883s.
[flaml.automl.logger: 10-16 08:11:15] {3192} INFO -  at 1.9s,	estimator lgbm's best error=4.0219e-02,	best estimator lgbm's best error=4.0219e-02
[flaml.automl.logger: 10-16 08:11:15] {3006} INFO - iteration 1, cur

RuntimeError: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.